# Fixed Income Quiz 3 - Credit Spread Models
## Merton (1974) and Merton (1976) Models

This notebook implements credit spread calculations using the Merton structural default models.

### Import Required Libraries

In [25]:
import sys
sys.path.append('.')

import numpy as np
import pandas as pd
from callable_debt_binomial import price_zero_coupon_debt_tree
from merton_models import merton_1974, merton_1976_with_jumps, black_scholes_call
from short_long_term_debt import short_long_term_debt_model


### Input Parameters

Define all parameters for the models below. Modify these values as needed for different scenarios.

In [26]:
# ============================================================================
# COMMON PARAMETERS (used in all implemented questions)
# ============================================================================

# Firm asset value at time 0
v = 100.0
# Long-term debt face value
M = 80.0
# Risk-free rate
r = 0.05
# Long-term maturity (years)
T = 4
# Asset volatility (Brownian motion component)
sigma = 0.25

theta = 1  # Theta is fixed to 1.0 per quiz instructions

# ============================================================================
# JUMP-SPECIFIC PARAMETERS (for Merton 1976 only)
# ============================================================================

# Jump intensity (Poisson process rate)
lambda_param = 0.4

# Mean jump magnitude E[e^J - 1]
mu_j = -0.1

# Jump volatility
sigma_j = 0.2

# Maximum number of terms in infinite sum
n_max = 100


# ============================================================================
# Q3/Q4 TREE PARAMETERS (callable, convertible, callable+convertible)
# ============================================================================

# rho is the redemption-price slope in K(t) = M * exp(-rho * (T - t)).
rho = 0.1
# Conversion ratio for convertible debt (creditors receive q * v upon conversion)
q = 0.6
# Number of CRR steps
n_steps_tree = 500


# ============================================================================
# Q5 PARAMETERS (short-term and long-term debt)
# ============================================================================

# Short-term debt face value
m = 40.0
# Short-term maturity (years)
t = 0.5


# just in case instead of mu_j we had the mu and sigma of the jump size distribution,
# we can calculate mu_j as follows:
# mu_j = np.exp(mu + 0.5 * sigma_j**2) - 1


# #just in case insted of q we had the shares n and N :
# n=60 # number of shares received upon conversion
# N=100 # number of shares outstanding before conversion
# q = n / (N + n)



#just in case: leverage L= D/ (D+ S) = D/(V) 

## Question 1: Zero-Coupon Debt with Merton (1974)

Calculate credit spreads using the classic Merton (1974) model with geometric Brownian motion.


In [27]:
# ============================================================================
# QUESTION 1: MERTON (1974) - ZERO-COUPON DEBT
# ============================================================================

# Calculate using Merton (1974) model
result_1974 = merton_1974(v=v, T=T, M=M, r=r, sigma=sigma)

print(f"\nModel Outputs:")
print(f"  Equity Value (S):                {result_1974['equity']:.6f}")
print(f"  Debt Value (D):                  {result_1974['debt']:.6f}")
print(f"  d1:                              {result_1974['d1']:.6f}")
print(f"  d2:                              {result_1974['d2']:.6f}")

print(f"\n>> CREDIT SPREAD:")
print(f"   Decimal form:                   {result_1974['credit_spread']:.6f}")
print(f"   Basis Points (bps):             {result_1974['credit_spread_bps']:.5f}")


Model Outputs:
  Equity Value (S):                38.898166
  Debt Value (D):                  61.101834
  d1:                              1.096287
  d2:                              0.596287

>> CREDIT SPREAD:
   Decimal form:                   0.017371
   Basis Points (bps):             173.71187


## Question 2: Zero-Coupon Debt with Merton (1976) - With Jumps

Calculate credit spreads using the Merton (1976) model that includes jump risk.


In [28]:
# ============================================================================
# QUESTION 2: MERTON (1976) - ZERO-COUPON DEBT WITH JUMPS
# ============================================================================

# Calculate using Merton (1976) model with jumps
result_1976 = merton_1976_with_jumps(
    v=v, 
    T=T, 
    M=M, 
    r=r, 
    sigma=sigma,
    lambda_param=lambda_param,
    mu_j=mu_j,
    sigma_j=sigma_j,
    n_max=n_max
)

print(f"\nModel Outputs:")
print(f"  Equity Value (S):                {result_1976['equity']:.6f}")
print(f"  Debt Value (D):                  {result_1976['debt']:.6f}")

print(f"\n>> CREDIT SPREAD:")
print(f"   Decimal form:                   {result_1976['credit_spread']:.6f}")
print(f"   Basis Points (bps):             {result_1976['credit_spread_bps']:.5f}")


Model Outputs:
  Equity Value (S):                40.694835
  Debt Value (D):                  59.305165

>> CREDIT SPREAD:
   Decimal form:                   0.024833
   Basis Points (bps):             248.32558


## Question 3: Zero-Coupon Callable Debt (Binomial Tree)

CRR tree initialized at the penultimate date.
Recursion uses `min(K(t), continuation)` with `K(t) = M exp(-rho (T - t))`.

In [29]:
# ============================================================================
# QUESTION 3: ZERO-COUPON CALLABLE DEBT (CRR BINOMIAL TREE)
# ============================================================================

def redemption_price(t_now, face_value):
    return face_value * np.exp(-rho * (T - t_now))

result_q3 = price_zero_coupon_debt_tree(
    v=v,
    M=M,
    T=T,
    r=r,
    sigma=sigma,
    n_steps=n_steps_tree,
    callable=True,
    convertible=False,
    q=q,
    redemption_fn=redemption_price,
    theta=theta,
)

print("\nModel Outputs:")
print(f"  Callable debt value (D):         {result_q3['debt']:.6f}")
print(f"  CRR up factor (u):               {result_q3['u']:.6f}")
print(f"  CRR down factor (d):             {result_q3['d']:.6f}")
print(f"  EMM up probability (pi):         {result_q3['p']:.6f}")

print("\n>> CREDIT SPREAD:")
print(f"   Callable debt spread (bps):     {result_q3['credit_spread_bps']:.5f}")


Model Outputs:
  Callable debt value (D):         53.625604
  CRR up factor (u):               1.022613
  CRR down factor (d):             0.977887
  EMM up probability (pi):         0.503355

>> CREDIT SPREAD:
   Callable debt spread (bps):     500.00000


## Question 4: Zero-Coupon Callable and Convertible Debt

CRR tree initialized at the penultimate date using convertible debt continuation.
Recursion uses `min(max(qv, continuation), max(qv, K(t)))` with `K(t) = M exp(-rho (T - t))`.

In [30]:
# ============================================================================
# QUESTION 4: ZERO-COUPON CALLABLE + CONVERTIBLE DEBT (CRR BINOMIAL TREE)
# ============================================================================

def redemption_price(t_now, face_value):
    return face_value * np.exp(-rho * (T - t_now))

result_q4 = price_zero_coupon_debt_tree(
    v=v,
    M=M,
    T=T,
    r=r,
    sigma=sigma,
    n_steps=n_steps_tree,
    callable=True,
    convertible=True,
    q=q,
    redemption_fn=redemption_price,
    theta=theta,
)

print("\nModel Outputs:")
print(f"  Callable+convertible debt (D):   {result_q4['debt']:.6f}")
print(f"  CRR up factor (u):               {result_q4['u']:.6f}")
print(f"  CRR down factor (d):             {result_q4['d']:.6f}")
print(f"  EMM up probability (pi):         {result_q4['p']:.6f}")

print("\n>> CREDIT SPREAD:")
print(f"   Callable+convertible spread:    {result_q4['credit_spread_bps']:.5f} bps")


Model Outputs:
  Callable+convertible debt (D):   60.000000
  CRR up factor (u):               1.022613
  CRR down factor (d):             0.977887
  EMM up probability (pi):         0.503355

>> CREDIT SPREAD:
   Callable+convertible spread:    219.20518 bps


## Question 5: Short-Term and Long-Term Debts

Calculate short-term and long-term credit spreads using Equations (9.24) and (9.25).

In [31]:
# ============================================================================
# QUESTION 5: SHORT-TERM AND LONG-TERM DEBTS
# ============================================================================

result_q5 = short_long_term_debt_model(
    v=v,
    m=m,
    t=t,
    M=M,
    T=T,
    r=r,
    sigma=sigma,
    theta=theta,
 )

print("\nModel Outputs:")
print(f"  Short-term debt value d_t:       {result_q5['short_term_debt']:.6f}")
print(f"  Long-term debt value D_T:        {result_q5['long_term_debt']:.6f}")

print("\n>> CREDIT SPREADS:")
print(f"   Short-term spread (bps):        {result_q5['short_term_spread_bps']:.5f}")
print(f"   Long-term spread (bps):         {result_q5['long_term_spread_bps']:.5f}")


Model Outputs:
  Short-term debt value d_t:       39.012394
  Long-term debt value D_T:        61.101833

>> CREDIT SPREADS:
   Short-term spread (bps):        0.00111
   Long-term spread (bps):         173.71191


In [32]:
# ============================================================================
# SHAREABLE SUMMARY: ALL INPUT PARAMETERS + ALL CREDIT SPREADS
# ============================================================================

print("\n==================== PARAMETER SUMMARY ====================")
param_names = [
    "v", "M", "r", "T", "sigma", "theta",
    "lambda_param", "mu_j", "sigma_j",
    "rho", "q", "n_steps_tree",
    "m", "t",
]

for name in param_names:
    value = globals().get(name, "<not set>")
    print(f"{name:>15}: {value}")

print("\n==================== CREDIT SPREADS (bps) ====================")
spreads = [
    ("Q1 Merton (1974)", globals().get("result_1974", {}).get("credit_spread_bps", "<not set>")),
    ("Q2 Merton (1976) jumps", globals().get("result_1976", {}).get("credit_spread_bps", "<not set>")),
    ("Q3 Callable debt", globals().get("result_q3", {}).get("credit_spread_bps", "<not set>")),
    ("Q4 Callable+convertible debt", globals().get("result_q4", {}).get("credit_spread_bps", "<not set>")),
    ("Q5 Short-term debt", globals().get("result_q5", {}).get("short_term_spread_bps", "<not set>")),
    ("Q5 Long-term debt", globals().get("result_q5", {}).get("long_term_spread_bps", "<not set>")),
]

for label, value in spreads:
    if isinstance(value, (int, float, np.floating)):
        print(f"{label:>35}: {float(value):.5f}")
    else:
        print(f"{label:>35}: {value}")


==================== PARAMETER SUMMARY ====================
              v: 100.0
              M: 80.0
              r: 0.05
              T: 4
          sigma: 0.25
          theta: 1
   lambda_param: 0.4
           mu_j: -0.1
        sigma_j: 0.2
            rho: 0.1
              q: 0.6
   n_steps_tree: 500
              m: 40.0
              t: 0.5

==================== CREDIT SPREADS (bps) ====================
                   Q1 Merton (1974): 173.71187
             Q2 Merton (1976) jumps: 248.32558
                   Q3 Callable debt: 500.00000
       Q4 Callable+convertible debt: 219.20518
                 Q5 Short-term debt: 0.00111
                  Q5 Long-term debt: 173.71191
